# Case Study 2 — Telecom Customer Churn Prediction

This is the guided Case Study 2 project, which predicts telecom customer churn. It is not the Netflix recommender in Case Study 1. The workflow below follows the brief: load and understand data, clean it, explore patterns, engineer features, split the data, train multiple models, combine them, evaluate, tune, and save outputs.

**Run order:** Start Jupyter in the project root (the folder containing `Datasets/`), open this notebook, then choose **Kernel → Restart Kernel and Run All Cells**. Detailed setup is in `README_Case_Study_2.md`.

**Outputs:** EDA charts, model metric table, confusion matrix, permutation importance, holdout review CSV, and saved soft-voting model in `artifacts/`. A probability is a risk estimate, never certainty or proof of cause. Do not automate adverse customer decisions from it.

## 1. Dataset field guide and expected contents

The inspected `Datasets/telco_churn.csv` has **5,043 rows and 22 columns**: 21 business fields plus an extra exported index. Read this table before running the modeling cells.

| Column | Exact meaning / unit | How used or cleaned |
|---|---|---|
| `Unnamed: 0` | Accidental row index from an earlier export; not customer behavior. | Dropped. Its values repeat. |
| `customerID` | Unique customer reference, e.g. `7590-VHVEG`. | Excluded from model features; kept for traceability in example review output. |
| `gender` | Recorded gender (`Female`, `Male`). | Category; one-hot encoded. Consider fairness/governance before production. |
| `SeniorCitizen` | Whether customer is a senior citizen; raw values mix `True`/`False` and `1`/`0`. | Normalize to `Yes`/`No`; category. |
| `Partner` | Whether customer has a partner. | Normalize boolean spellings; category. |
| `Dependents` | Whether customer has dependents. | Normalize boolean spellings; category. |
| `tenure` | Customer relationship length in **months**, not years. | Numeric; zero-month rows are excluded per case-study cleaning guidance because their total-charge history is blank. |
| `PhoneService` | Whether phone service is active. | Normalize boolean spellings; category. |
| `MultipleLines` | Multi-line phone service status. | Blank with no phone service -> `No phone service`; unexplained blank -> `Unknown`. |
| `InternetService` | DSL, Fiber optic, or no internet (raw category `No`). | Category; `No` remains distinct from other add-on answers. |
| `OnlineSecurity` | Online security add-on status. | Blank when InternetService=`No` -> `No internet service`; other blanks -> `Unknown`. |
| `OnlineBackup` | Online backup add-on status. | Same internet-aware blank handling. |
| `DeviceProtection` | Device-protection add-on status. | Same internet-aware blank handling. |
| `TechSupport` | Tech-support add-on status. | Same internet-aware blank handling. |
| `StreamingTV` | Streaming TV add-on status. | Same internet-aware blank handling. |
| `StreamingMovies` | Streaming movies add-on status. | Same internet-aware blank handling. |
| `Contract` | Month-to-month, one-year, or two-year contract. | Category; one-hot encoded. |
| `PaperlessBilling` | Whether billing is paperless. | Normalize boolean spellings; category. |
| `PaymentMethod` | Payment method (electronic check, mailed check, automatic bank transfer, automatic credit card). | Category; one-hot encoded. |
| `MonthlyCharges` | Current recurring monthly charge (currency units unspecified in CSV). | Numeric; training median imputation and standardization. |
| `TotalCharges` | Accumulated charges (same unspecified currency units); stored as text in raw CSV. | Parse to number. Empty zero-tenure history is removed with zero-tenure rows; other missing values are training-imputed. |
| `Churn` | Whether customer left: Yes or No. | Prediction target. `True`/`False` normalized to `Yes`/`No`; missing labels excluded. |

**Keep these meanings distinct:** `No` means no to an applicable service; `No phone service`/`No internet service` mean the customer does not have the underlying service; `Unknown` means the CSV does not explain the blank. The inspected file has 269 `MultipleLines` blanks (all no-phone rows), 651 blanks in each of six internet add-ons (all `InternetService=No`), one missing churn label, and 8 zero-tenure rows. Rerun the audit if the dataset changes.

## 2. Imports, reproducibility, and data path

In [ ]:
from pathlib import Path
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, ConfusionMatrixDisplay)
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC

warnings.filterwarnings("ignore", category=FutureWarning)
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid")
DATA_PATH = Path("Datasets") / "telco_churn.csv"
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Missing {DATA_PATH.resolve()}. Start Jupyter in the project root or edit DATA_PATH.")
print("Reading:", DATA_PATH.resolve())

## 3. Load and audit the raw CSV

The audit checks dimensions, names, inferred data types, missing values, distinct values, and target labels before cleaning. `raw` remains untouched so you can compare it with the cleaned table.

In [ ]:
raw = pd.read_csv(DATA_PATH)
print("Shape (rows, columns):", raw.shape)
print("Columns:", raw.columns.tolist())
display(raw.dtypes.astype(str).rename("dtype").to_frame())
display(raw.isna().sum().rename("missing").to_frame())
display(raw.nunique(dropna=True).rename("distinct non-null values").to_frame())
display(raw.head())
if "Churn" in raw: display(raw["Churn"].value_counts(dropna=False).rename("raw rows").to_frame())

## 4. Clean and normalize, with reasons

- Remove the exported index; it is not a feature. Keep `customerID` outside of model inputs.
- Trim text and map boolean spellings (`True`, `False`, `1`, `0`, `Yes`, `No`) to canonical `Yes`/`No`. This prevents duplicate encodings of the same answer.
- Parse tenure and charges as numeric. Invalid `TotalCharges` becomes missing, not zero.
- Resolve a blank service answer only if another field explains it: no phone -> `No phone service`; no internet -> `No internet service`. Other categorical blanks become `Unknown`.
- Remove rows whose target is missing because supervised training requires a known answer. Remove zero-tenure rows as specified by the case study; this file has blank total-charge history for those rows.
- Remaining numeric missing values are imputed later using only training data.
- Add `AvgMonthlySpendToDate = TotalCharges / tenure`, an approximate accumulated charge per tenure month. This is not a bill prediction and should be interpreted cautiously.

In [ ]:
df = raw.copy()
start_rows = len(df)
df = df.drop(columns=[c for c in df if c.lower().startswith("unnamed:")])
required = {"customerID", "Churn", "tenure", "MonthlyCharges", "TotalCharges"}
if not required.issubset(df.columns): raise ValueError(f"Missing columns: {sorted(required-set(df.columns))}")
for c in df.select_dtypes(include="object").columns:
    df[c] = df[c].map(lambda v: v.strip() if isinstance(v, str) else v)
canon = {"true":"Yes", "false":"No", "1":"Yes", "0":"No", "yes":"Yes", "no":"No"}
for c in df.select_dtypes(include="object").columns:
    df[c] = df[c].map(lambda v: canon.get(v.lower(), v) if isinstance(v, str) else v)
for c in ["tenure", "MonthlyCharges", "TotalCharges"]: df[c] = pd.to_numeric(df[c], errors="coerce")
df["Churn"] = df["Churn"].map({"Yes":"Yes", "No":"No"})
removed_missing_target = int(df["Churn"].isna().sum())
df = df.loc[df["Churn"].notna()].copy()
removed_zero_tenure = int(df["tenure"].eq(0).sum())
df = df.loc[~df["tenure"].eq(0)].copy()
if {"MultipleLines", "PhoneService"}.issubset(df.columns):
    m = df["MultipleLines"].isna() & df["PhoneService"].eq("No")
    df.loc[m, "MultipleLines"] = "No phone service"
addons = ["OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"]
for c in addons:
    if c in df and "InternetService" in df:
        m = df[c].isna() & df["InternetService"].eq("No")
        df.loc[m, c] = "No internet service"
for c in df.select_dtypes(include="object").columns:
    if c not in {"customerID", "Churn"}: df[c] = df[c].fillna("Unknown")
df["AvgMonthlySpendToDate"] = df["TotalCharges"] / df["tenure"]
print(f"Rows at start: {start_rows:,}; removed missing target: {removed_missing_target:,}; removed zero tenure: {removed_zero_tenure:,}; cleaned rows: {len(df):,}")
print("Remaining missing values (numeric values are imputed inside the training pipeline):")
display(df.isna().sum().rename("missing").to_frame())
display(df["Churn"].value_counts().rename("clean target count").to_frame())

## 5. EDA: churn rate and requested relationships

The overall rate is the count of `Churn=Yes` divided by all labeled clean rows. Group charts show *within-group churn rates* rather than raw counts, so groups of different sizes can be compared. These are associations, not evidence that a feature causes churn. Small groups can have unstable rates.

In [ ]:
df["churn_flag"] = df["Churn"].eq("Yes").astype(int)
print(f"Observed churn rate: {df['churn_flag'].mean():.1%}")
fig, ax = plt.subplots(2, 2, figsize=(13, 9))
sns.countplot(data=df, x="Churn", order=["No", "Yes"], ax=ax[0,0]); ax[0,0].set_title("Customer count by churn label")
sns.lineplot(data=df.groupby("tenure", as_index=False)["churn_flag"].mean(), x="tenure", y="churn_flag", ax=ax[0,1]); ax[0,1].set(title="Churn rate by tenure (months)", ylabel="Churn rate")
for axis, col in [(ax[1,0],"Contract"),(ax[1,1],"PaymentMethod")]:
    rate = df.groupby(col, as_index=False)["churn_flag"].mean().sort_values("churn_flag")
    sns.barplot(data=rate, x="churn_flag", y=col, ax=axis); axis.set(title=f"Churn rate by {col}", xlabel="Within-group churn rate")
plt.tight_layout(); plt.show()
print("EDA note: document patterns from your plots; do not claim causation.")

## 6. Features, target, and train/test split

`Churn` becomes `1=Yes` and `0=No`. `customerID` and the target are excluded from predictors. The derived tenure bands support segmentation; their boundaries are 1–12, 13–24, 25–48, and 49–72 months. An 80/20 stratified split preserves the churn proportion. The held-out 20% is not used for fitting, imputation, encoding, scaling, or tuning.

In [ ]:
y = df["Churn"].eq("Yes").astype(int)
X = df.drop(columns=["Churn", "customerID", "churn_flag"], errors="ignore").copy()
X["TenureGroup"] = pd.cut(X["tenure"], bins=[0,12,24,48,72], labels=["1-12 months","13-24 months","25-48 months","49-72 months"], include_lowest=True).astype("object")
numeric_features = [c for c in ["tenure","MonthlyCharges","TotalCharges","AvgMonthlySpendToDate"] if c in X]
categorical_features = [c for c in X if c not in numeric_features]
for c in numeric_features: X[c] = pd.to_numeric(X[c], errors="coerce")
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=.20,random_state=RANDOM_STATE,stratify=y)
print("Input columns:", X.columns.tolist())
print("Numeric:", numeric_features, "\nCategorical:", categorical_features)
print(f"Train={len(X_train):,} rows ({y_train.mean():.1%} churn); test={len(X_test):,} rows ({y_test.mean():.1%} churn)")

## 7. Preprocessing pipelines

Numeric columns use median imputation and standard scaling; scaling is useful for Logistic Regression and SVM. Categorical columns use most-frequent imputation and one-hot encoding. Unknown future categories are ignored by the encoder rather than causing a crash. The transformations are inside each pipeline, so cross-validation learns them only from that fold's training portion. This prevents test-set leakage.

In [ ]:
num_pipe = Pipeline([("imputer",SimpleImputer(strategy="median")),("scale",StandardScaler())])
cat_pipe = Pipeline([("imputer",SimpleImputer(strategy="most_frequent")),("onehot",OneHotEncoder(handle_unknown="ignore"))])
preprocess = ColumnTransformer([("numeric",num_pipe,numeric_features),("categorical",cat_pipe,categorical_features)])
def make_pipe(model): return Pipeline([("preprocess",clone(preprocess)),("model",model)])

## 8. Fit individual models and combine them

Logistic Regression is a regularized linear classifier; Random Forest averages many decision trees; SVM separates classes using a margin and probability estimates. Class weights make the first two models pay extra attention to the minority class without fabricating data.

The **soft-voting ensemble** is the combined model requested in Case Study 2: it averages the three estimated churn probabilities. A threshold of 0.50 makes its final Yes/No call. It may not outperform every individual model; report measured results, not an assumed winner.

In [ ]:
models = {
 "Logistic Regression":make_pipe(LogisticRegression(max_iter=2000,class_weight="balanced",random_state=RANDOM_STATE)),
 "Random Forest":make_pipe(RandomForestClassifier(n_estimators=250,min_samples_leaf=2,class_weight="balanced",random_state=RANDOM_STATE,n_jobs=-1)),
 "SVM":make_pipe(SVC(C=1.0,kernel="rbf",probability=True,class_weight="balanced",random_state=RANDOM_STATE))
}
for name, model in models.items(): model.fit(X_train,y_train); print("Fitted",name)
ensemble = VotingClassifier(estimators=[(k.lower().replace(" ","_"),clone(v)) for k,v in models.items()],voting="soft",n_jobs=-1)
ensemble.fit(X_train,y_train)
print("Fitted soft voting ensemble")

## 9. Holdout evaluation

Accuracy is total correctness. Churn precision is the share of flagged customers who truly churned; churn recall is the share of churners found; F1 balances precision and recall. ROC-AUC measures probability ranking across thresholds, not the quality of the one threshold alone. A false positive can waste a retention contact; a false negative misses a customer who leaves. Choose a threshold with the business based on intervention cost, customer impact, and team capacity.

In [ ]:
fitted = {**models,"Soft Voting Ensemble":ensemble}
rows=[]; preds={}; probs={}
for name,model in fitted.items():
    pred=model.predict(X_test); prob=model.predict_proba(X_test)[:,1]; preds[name]=pred; probs[name]=prob
    rows.append({"Model":name,"Accuracy":accuracy_score(y_test,pred),"Churn precision":precision_score(y_test,pred,zero_division=0),"Churn recall":recall_score(y_test,pred,zero_division=0),"Churn F1":f1_score(y_test,pred,zero_division=0),"ROC-AUC":roc_auc_score(y_test,prob)})
metrics=pd.DataFrame(rows).sort_values("Churn F1",ascending=False).reset_index(drop=True)
display(metrics.round(3))
print(classification_report(y_test,preds["Soft Voting Ensemble"],target_names=["Stayed (0)","Churned (1)"],zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test,preds["Soft Voting Ensemble"],display_labels=["Stayed","Churned"],cmap="Blues",values_format="d"); plt.title("Ensemble holdout confusion matrix"); plt.show()

## 10. Tune Random Forest on training data only

Grid search tries 8 combinations of tree count, depth, and minimum leaf size. Stratified 3-fold cross-validation selects by churn F1. The test set remains untouched until the tuned model's final evaluation. This is a compact demonstration search; larger searches cost more time.

In [ ]:
rf=make_pipe(RandomForestClassifier(class_weight="balanced",random_state=RANDOM_STATE,n_jobs=-1))
params={"model__n_estimators":[150,300],"model__max_depth":[None,12],"model__min_samples_leaf":[1,3]}
cv=StratifiedKFold(n_splits=3,shuffle=True,random_state=RANDOM_STATE)
search=GridSearchCV(rf,params,scoring="f1",cv=cv,n_jobs=-1,refit=True)
search.fit(X_train,y_train); tuned_rf=search.best_estimator_
tuned_pred=tuned_rf.predict(X_test); tuned_prob=tuned_rf.predict_proba(X_test)[:,1]
print("Best training CV F1:",round(search.best_score_,3),"; parameters:",search.best_params_)
tuned_row={"Model":"Tuned Random Forest","Accuracy":accuracy_score(y_test,tuned_pred),"Churn precision":precision_score(y_test,tuned_pred,zero_division=0),"Churn recall":recall_score(y_test,tuned_pred,zero_division=0),"Churn F1":f1_score(y_test,tuned_pred,zero_division=0),"ROC-AUC":roc_auc_score(y_test,tuned_prob)}
metrics=pd.concat([metrics,pd.DataFrame([tuned_row])],ignore_index=True).sort_values("Churn F1",ascending=False).reset_index(drop=True)
display(metrics.round(3))

## 11. Explain which fields the ensemble relies on

Permutation importance shuffles one original input at a time and measures the resulting drop in holdout F1. It describes this model's reliance, not causation; correlated inputs can share importance. Three repeats are used to keep runtime moderate.

In [ ]:
pi=permutation_importance(ensemble,X_test,y_test,scoring="f1",n_repeats=3,random_state=RANDOM_STATE,n_jobs=-1)
importance=pd.DataFrame({"Feature":X_test.columns,"Mean F1 drop":pi.importances_mean,"Variation":pi.importances_std}).sort_values("Mean F1 drop",ascending=False)
display(importance.head(15).round(4))
sns.barplot(data=importance.head(15),x="Mean F1 drop",y="Feature",color="steelblue"); plt.title("Permutation importance (ensemble, holdout)"); plt.tight_layout(); plt.show()

## 12. Create a customer-review list and save deliverables

The holdout review list contains true labels only for evaluation. A future scoring file must omit `Churn`. The threshold is a configurable baseline, not a policy recommendation. The saved pipeline includes its preprocessing. Only load joblib/pickle models created by a trusted source.

In [ ]:
threshold=0.50
review=pd.DataFrame({"customerID":df.loc[X_test.index,"customerID"].values,"actual_Churn_for_evaluation_only":y_test.values,"predicted_churn_probability":ensemble.predict_proba(X_test)[:,1]})
review["flag_at_threshold"]=review["predicted_churn_probability"]>=threshold
review=review.sort_values("predicted_churn_probability",ascending=False)
display(review.head(10))
print(f"At threshold {threshold:.2f}, flagged {review.flag_at_threshold.sum():,} of {len(review):,} holdout customers.")
ARTIFACTS=Path("artifacts"); ARTIFACTS.mkdir(exist_ok=True)
joblib.dump(ensemble,ARTIFACTS/"case_study_2_soft_voting_model.joblib")
metrics.to_csv(ARTIFACTS/"model_metrics.csv",index=False)
review.to_csv(ARTIFACTS/"holdout_risk_review.csv",index=False)
importance.to_csv(ARTIFACTS/"permutation_importance.csv",index=False)
print("Saved:",*[str(p) for p in ARTIFACTS.iterdir()],sep="\n- ")

## 13. Score one new record (template)

A new record must have the same feature columns as `X`; it must not include `Churn` or `customerID`. The example borrows one feature row from the training set only to demonstrate the call. Replace it with an authorized record and do not put personal data in an unsecured notebook.

In [ ]:
loaded=joblib.load(ARTIFACTS/"case_study_2_soft_voting_model.joblib")
example=X_train.iloc[[0]].copy()
score=float(loaded.predict_proba(example)[0,1])
print(f"Illustrative estimated churn probability: {score:.1%}. Risk estimate, not certainty or causal explanation.")

## 14. Findings to complete and submission checklist

After running, write down: (1) rows removed and why, missing-value decisions, and churn share; (2) the actual EDA patterns and their group sizes; (3) holdout precision, recall, F1, ROC-AUC for all models, including the ensemble and tuned forest; (4) false-positive/false-negative tradeoff and threshold rationale; (5) evidence-based retention suggestions; (6) limitations. Do not reuse a score from the old README or claim results this run did not produce.

**Limitations:** no dates, interventions, customer support history, or future outcomes are present. A random holdout is not a temporal future test; correlations do not prove cause; high risk does not mean certain churn. Before operational use, validate on later data, review fairness/calibration, monitor drift, and test interventions.

**Capstone checklist:** CSV loaded and audited; cleaning and EDA explained; features engineered; stratified split; Logistic Regression, Random Forest, SVM, and a soft-voting combined model fitted; accuracy/precision/recall/F1/ROC-AUC and confusion matrix reported; Random Forest tuned; importance and saved artifact produced. Complete team-specific capstone forms, final documentation, and PowerPoint with your trainer.